In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
from langchain_anthropic import ChatAnthropic
llm = ChatAnthropic(model="claude-sonnet-5-5")

## Problem

- We can only return the string.
_ What if we want to return a different data type in response.

In [4]:
text = "my name is Afraz. I am living In Rouen France.\
     I born in Gujrat, Pakistan. I am 30 years old."

res = llm.invoke(f"Find the name, current place,  birth place, and age in {text}")
print(res.content)

**Extracted Information**

| Item          | Details               |
|---------------|-----------------------|
| **Name**      | Afraz                 |
| **Current Place** | Rouen, France      |
| **Birth Place**   | Gujrat, Pakistan   |
| **Age**           | 30 years old       |


# Solution

- We can use `pydentic` for structured response

In [5]:
from pydantic import BaseModel, Field # field validation

class Structured_res(BaseModel):
    name: str = Field(description =  "User name")
    living_place: str = Field(description = "Current Place of living")
    bith_place: str = Field(description = "Place of birth") 
    age: int = Field("Age")

In [6]:
# to print structured output we use `with_structured_output` class before `invoke`
structured_llm = llm.with_structured_output(Structured_res)

res = structured_llm.invoke(f"Find the name, Birth Place, Living Place and Age from {text}")

In [7]:
res

Structured_res(name='Afraz', living_place='Rouen, France', bith_place='Gujrat, Pakistan', age=30)

In [8]:
res.name

'Afraz'

In [9]:
res.age

30

In [10]:
print(res.model_dump())

{'name': 'Afraz', 'living_place': 'Rouen, France', 'bith_place': 'Gujrat, Pakistan', 'age': 30}


# Make a bit complex

In [11]:
# get only one movie name
class Movie(BaseModel):
    name: str = Field(description= "Name of Movie")
    year: int = Field(description= "Movie release year")
    idea: str = Field(description= "Movie's main idea")

movie_llm = llm.with_structured_output(Movie)

res = movie_llm.invoke("Tell me the best French movie for French language learning purpose.")
print(res)

None


In [12]:
# getting multiple movies name
# to get the more movies name we will wrap the Movies() in List
from typing import List

class Movies(BaseModel):
    movies: List[Movie] = Field(description = "List of Movies")

movies_llm = llm.with_structured_output(Movies)
res = movies_llm.invoke("List 3 best French movies for French language learning purpose.")

print(res)

movies=[Movie(name='Amélie', year=2001, idea='A whimsical romantic comedy set in Paris, featuring clear, everyday dialogue and cultural references, ideal for beginners to intermediate learners.'), Movie(name='Intouchables, Les', year=2011, idea='A heart‑warming drama about friendship and personal growth, with natural conversational French and a mix of formal and informal speech, suitable for intermediate learners.'), Movie(name='La Haine', year=1995, idea='A gritty urban drama that explores social issues in the French suburbs, offering exposure to colloquial slang and fast‑paced dialogue for advanced learners.')]
